I want you to design a class-based backtesting system for cross-sectional ML stock selection. Do not write Python code yet. First, fully understand and evaluate the structure and make sure the workflow is logically consistent, memory-aware, modular, and free of look-ahead bias under the timing convention below.

Use a class named `Quant_Strategy_ML`.

Important timing convention:
- The factor row with index t is available at the beginning of trading day t.
- The return row with index t is the realized return for the current period ending on day t.
- For example, the return stored at row `2020-01-01` is the aggregated return over the period ending on `2020-01-01`.
- We can observe factor_t in the morning of day t.
- We can observe return_t near the end of trading day t.
- We retrain the model at night on day t using data up to row t.
- The updated model is first used starting on day t+1.
- Under this convention, using factor_t and return_t together for stock screening, factor screening, Rank IC, Rank IR, and model training does not create look-ahead bias, as long as the fitted model is only used from t+1 onward.

General requirements:
- Use classes and methods only.
- The system should be modular and readable.
- Factor files and stock return data are stored as pickle files.
- Each factor file is a panel with row index = date and columns = stock permnos.
- The stock return file has the same panel structure: row index = date and columns = stock permnos.
- For the current training window, the computer is capable of loading all factor panels into memory at once, so you may assume this is allowed.
- However, the design should still avoid unnecessary duplicated objects and unnecessary full-history reloading.

1. Class initialization
The class `Quant_Strategy_ML` should be initialized with a dictionary containing all user inputs and configuration settings.

The user must first provide:
- `Factor_Path`: folder path containing factor pickle files
- `Return_Path`: file path containing the stock return pickle file

The class should:
- store the config
- initialize key attributes
- prepare internal containers for factors, labels, predictions, portfolio weights, and performance results

2. Training/testing window setup
Create a method called `set_train_test_window()`.

This method should:
- load the stock return panel as `self.return_data`
- extract the ordered trading date list as `self.trading_date_list`
- accept user inputs:
  - `start_period`
  - `end_period`
  - `train_window_periods`
  - `train_interval_periods`
- build rolling train/test windows
- output:
  - `self.train_period_lists`
  - `self.test_period_lists`

Each element in `self.train_period_lists` and `self.test_period_lists` represents one rolling train/test window pair.

3. Rolling-window workflow
For each pair of train/test windows in `self.train_period_lists` and `self.test_period_lists`, repeat the full workflow below.

4. Load training factors
Create a method called `load_factors()`.

This method should:
- load all factor data for the current training window from `Factor_Path`
- reindex each factor panel to the current raw training window
- store them in a dictionary:
  - `self.factors_train_dic`

Dictionary structure:
- key = factor name
- value = corresponding factor dataframe

Because the machine can handle it, it is acceptable to load all factor panels for the current training window before proceeding.

5. Stock availability screening
Create a method called `available_stock()`.

Purpose:
Select stocks with enough valid data during the current training window.

User inputs:
- `stock_valid_period_frac`
- `stock_valid_factor_frac`
- `ret_valid_period_frac`

Selection logic:
- A stock is valid if at least `stock_valid_factor_frac` of factors have non-missing values for at least `stock_valid_period_frac` of the training periods
- The stock must also have at least `ret_valid_period_frac` of non-missing return observations from `self.return_data`

Output:
- `self.stock_list`

6. Reindex to selected stocks
Reindex both:
- factor data
- stock return data
to only include the chosen stocks.

Print the number of available stocks.

7. Factor selection
Create a method called `factor_selection()`.

Important rule:
- Factor selection is based on the return data directly, not on custom labels.
- Therefore, `factor_selection()` should come before `label_construction()`.

User inputs:
- `factor_valid_period_frac`
- `factor_valid_stock_frac`
- `factor_filter` dictionary

First-stage factor validity screen:
- Keep factors such that at least `factor_valid_stock_frac` of stocks have at least `factor_valid_period_frac` of periods with non-missing values

Second-stage factor filter:
- The user can provide a dictionary such as:
  - `{Rank_IC: {"threshold": IC_threshold}, Rank_IR: {"threshold": IR_threshold, "Rank_threshold": 100}}`
- `Rank_IC` and `Rank_IR` are algorithms that score each factor using the return data
- These metrics are computed on the current training window using the selected stocks and the return panel
- Factors must satisfy all threshold conditions
- If ranking is requested, rank the remaining factors and keep only the top factors

Keep only selected factors in `self.factors_train_dic` and drop the others.

Output:
- list of selected factors
- updated `self.factors_train_dic`

Print the number of valid factors.

8. Label construction
Create a method called `label_construction()`.

This method should allow the user to specify:
- a label function
- label parameters

Examples:
- `current_period_return()`
- `future_return(n)`
- `future_spread(n)`

The method should output:
- `self.label`
- `self.label_future_periods`

Important rule:
- Labels are used for model training, not for the earlier IC/IR-based factor selection step.
- Therefore, the choice of label affects the final model training sample, but does not change the earlier stock selection and factor selection logic unless explicitly redesigned later.

9. Final training window after label definition
After label construction, create the final model-training window.

Rules:
- If the label is current-period return aligned with the timing convention above, then no future exclusion is needed.
- If the label uses future periods, then exclude the last relevant periods from the training window because those rows do not have fully observable labels at that time.

Store the result as:
- `self.future_excluded_train_window`

10. Valid training period selection
Create a method called `valid_train_period()`.

Purpose:
Choose the final training dates that have enough valid information for model fitting.

User inputs:
- `valid_stock_frac`
- `valid_factor_frac`

A valid training period should have:
- at least `valid_factor_frac` of selected factors
- and each of those factors should have at least `valid_stock_frac` of stocks with non-missing values

This step should be applied to the final training window that is valid after label construction.

Output:
- `self.adjust_training_window`

11. Missing value handling
11a. Labels
- Reindex labels to `self.adjust_training_window`
- Do not fabricate labels beyond their observable range
- Missing labels should be dropped later when forming the final training sample

11b. Factors
- Fill factor NaN values using either:
  - `fill_median_ts()`
  - `fill_median_cs()`
- Then reindex the selected factor data to `self.adjust_training_window`

11c. Logging
Print:
- training window start date
- training window end date
- total number of training periods

12. Standardization
Allow the user to choose whether to standardize the factors.

If yes:
- apply z-score standardization factor by factor
- fit standardization parameters using training data only
- save those parameters for use in testing

13. Training data preparation
Create a method called `training_preparation()`.

Purpose:
Convert panel data into model-ready training data.

Default output:
- `self.feature_x`: rows = period × stock, columns = selected factors
- `self.label_y`: one-column label vector

Rules:
- align features and labels correctly
- drop rows where the label is missing
- preserve stock/date mapping if needed for later diagnostics

If different model families require different data shapes later, keep the design extensible.

14. Model training
Create a method called `training_function()`.

This method should allow the user to specify:
- model type (for example: linear model, tree model, LSTM, etc.)
- hyperparameter dictionary

It should train using:
- `self.feature_x`
- `self.label_y`

After training, save the fitted model and any preprocessing objects needed for testing.

15. Testing stage
For each period or date in the current test window, do the following.

15a. Load test factors
- Load factor data only for the selected factors from training
- Reindex to the current test period/date

15b. Test stock selection
Create a method called `select_test_stock()`.

User input:
- `valid_feature_frac`

Selection rule:
- choose stocks with at least `valid_feature_frac` of selected features non-missing on the test date

15c. Test preprocessing
- Apply the same factor filling logic if needed
- Apply the training-fitted standardization parameters if standardization was used
- Never refit preprocessing using test data

15d. Prediction
- Generate predicted label values for the selected stocks on the test date
- Store them in a panel structure

Print:
- test window start date
- test window end date
- total number of test periods
- number of stocks
- number of factors

16. Predicted label panel
Concatenate all predicted values across test dates/windows into:
- `self.predicted_labels`

This panel may contain NaN values because the selected stock set can differ across dates.

17. Predicted value adjustment
Create a method called `adjust_pred_value()`.

User input:
- `process_dictionary`

Example:
- `{moving_avg: avg_length, neutralization: feature_list}`

Possible adjustments:
1. `moving_avg(avg_length)`
- smooth predicted values using past predictions in a stock-level way over time

2. `neutralization(feature_list)`
- for each date, cross-sectionally regress the predicted value on the specified features
- use the regression residual as the neutralized predicted value

Output:
- `self.adjusted_predicted_labels`

18. Portfolio formation
Create a method called `form_portfolio()`.

User inputs:
- `number_stock_pick`
- `exchange_frac`

Portfolio logic:
- On the first test period, rank stocks by predicted value
- Long the top `number_stock_pick` stocks
- Short the bottom `number_stock_pick` stocks
- Use equal weights within the long side and within the short side

For each later period:
- Replace only the worst `exchange_frac` fraction of currently held long positions with new higher-ranked candidates not already held
- Replace only the worst `exchange_frac` fraction of currently held short positions with new lower-ranked candidates not already held
- Rebalance to equal weights after the update

Output:
- `self.portfolio_weights`

19. Backtesting
Create a method called `backtest()`.

Inputs:
- return panel
- portfolio weights
- `transaction_cost_rate`

This method should calculate:
- portfolio return for each period
- turnover rate for each period
- transaction-cost-adjusted return
- annualized return
- Sharpe ratio
- maximum drawdown
- average turnover rate

Performance requirement:
- use `numba` to efficiently calculate portfolio return and turnover on NumPy arrays
- use pandas mainly for alignment and panel handling

20. Visualization and reporting
Use `plotly` to create output figures and tables.

At minimum include:
- cumulative return curve
- drawdown curve
- turnover curve
- yearly performance table

The yearly performance table should include:
- annual return
- Sharpe ratio
- maximum drawdown
- average turnover rate

Additional implementation requirements:
- keep method names exactly as specified above
- keep the class readable and modular
- avoid unnecessary duplicate objects in memory
- make the design robust enough to support different label definitions, factor filters, ML models, and portfolio rules

# Project Prompt (Revised Time Convention)

Build a Python class for a cross-sectional ML stock-selection backtest.

## Data Structure

- The factor files are stored as pickle files in a factor folder.
- Each factor file is a wide panel:
  - row index = monthly timestamp
  - columns = stock `permno`
  - values = factor values
- The timestamps are stored as first-of-month dates such as `2020-01-01`, but for the OSP factor data they represent the **signal for that calendar month available by month-end**.
- The stock return file is stored as a pickle file called `monthly_end_return.pkl`.
- `monthly_end_return.pkl` has the same wide-panel structure:
  - row index = monthly timestamp
  - columns = stock `permno`
  - values = realized stock return for that calendar month
- The return timestamps are also stored as first-of-month dates, but row `2020-02-01` means the **realized return over February 2020**, i.e. from January month-end to February month-end.

## Correct Time Convention

- A factor row with index `t` is the signal for month `t`, known by the end of month `t`.
- A return row with index `t` is the realized return during month `t`.
- Therefore, a factor at row `t` can only be used to predict and trade the return at row `t+1`.
- Example:
  - factor row `2020-01-01` = January 2020 signal, known by January 2020 month-end
  - return row `2020-02-01` = realized February 2020 return
  - so January signal should be aligned with February return

## System Requirements

The class should be initialized with a config dictionary.

Important config fields include:

- `Factor_Path`
- `Return_Path`
- `start_period`
- `end_period`
- `train_window_periods`
- `train_interval_periods`
- `stock_valid_period_frac`
- `stock_valid_factor_frac`
- `ret_valid_period_frac`
- `factor_valid_period_frac`
- `factor_valid_stock_frac`
- `factor_filter`
- `label_function`
- `label_params`
- `valid_stock_frac`
- `valid_factor_frac`
- `fill_method`
- `standardize`
- `model_class`
- `model_params`
- `valid_feature_frac`
- `process_dictionary`
- `number_stock_pick`
- `exchange_frac`
- `transaction_cost_rate`
- `periods_per_year`

Also include:

- `signal_delay_periods = 1`

## Rolling Window Logic

Implement a method `set_train_test_window()`.

- Load the stock return panel as `self.return_data`
- Extract the trading date list as `self.trading_date_list`
- Use `start_period` and `end_period` to define the monthly signal range
- Use rolling windows:
  - training window length = `train_window_periods`
  - testing window length = `train_interval_periods`
- Save:
  - `self.train_period_lists`
  - `self.test_period_lists`

## Factor Loading

Implement `load_factors()`.

- Load all factor pickle files for the current training window
- Save them in `self.factors_train_dic`

## Available Stock Selection

Implement `available_stock()`.

- Select stocks with enough non-missing factor data and enough non-missing return data
- Because factors at `t` map to returns at `t+1`, stock return validity must be checked on the **aligned return panel shifted by one period**
- Save the selected stock list as `self.stock_list`

## Reindexing

After stock selection:

- Reindex all factor panels to `self.stock_list`
- Reindex return data to `self.stock_list`

## Factor Selection

Implement `factor_selection()`.

- First apply factor availability filtering
- Then compute `Rank_IC` and `Rank_IR`
- `Rank_IC` and `Rank_IR` must be computed using:
  - factor row `t`
  - aligned return row `t+1`
- Apply the user-defined filtering rules
- Save:
  - `self.selected_factors`
  - `self.selected_factor_metrics`

## Label Construction

Implement `label_construction()`.

Supported labels should include:

- `current_period_return`
- `future_return(n)`
- `future_spread(n)`

Timing rule:

- `current_period_return` means the aligned next tradable period return, so under this convention it should use delay `1`
- `future_return(n)` should compound returns from `t+1` through `t+n`
- `future_spread(n)` should use the same future alignment

Save:

- `self.label`
- `self.label_future_periods`

If the label uses future periods, exclude the final unobservable training dates accordingly.

## Valid Training Dates

Implement `valid_train_period()`.

- Keep only training dates where enough stocks and enough selected factors are non-missing
- This check should be applied on raw factor data before filling missing values

## Missing Value Handling

Implement missing value support.

Allowed methods:

- cross-sectional median fill
- time-series median fill
- no fill

Save any training-only fill statistics needed for test-time processing.

## Standardization

Implement training-only standardization.

- Fit mean/std on training data only
- Apply the saved parameters to test data

## Training Data Preparation

Implement `training_preparation()`.

- Stack the selected factor panels into a flat feature matrix
- Stack labels into a flat target vector
- Remove invalid rows with missing labels or missing features

## Model Training

Implement `training_function()`.

- Instantiate the model from `model_class` and `model_params`
- Fit the model on the prepared training data

## Test Prediction

Implement `_test_window()`.

- For each test date `t`, build the factor cross section at signal month `t`
- Select valid stocks
- Apply test-time filling and standardization using training-only objects
- Predict scores for that signal month
- Save predictions in `self.predicted_labels`

## Prediction Post-Processing

Implement optional post-processing.

Examples:

- moving average
- neutralization

Save processed predictions as `self.adjusted_predicted_labels`

## Portfolio Formation

Implement `form_portfolio()`.

- Form a long-short equal-weight portfolio
- Use `number_stock_pick`
- Use `exchange_frac` to control gradual turnover
- Portfolio weights formed on signal date `t` should be stored in `self.portfolio_weights`

## Backtesting

Implement `backtest()`.

- A portfolio formed from signal row `t` must be executed on return row `t+1`
- Therefore, execution weights must be shifted forward by `signal_delay_periods`
- Compute:
  - portfolio return
  - transaction-cost-adjusted return
  - annualized return
  - Sharpe ratio
  - max drawdown
  - average turnover
- Use efficient NumPy / numba-style logic for the backtest core

## Boundary Rule

- `start_period` and `end_period` define the requested backtest sample
- Report realized backtest returns only for dates within the requested evaluation range unless explicitly stated otherwise

## Visualization

Implement `visualize()`.

- Plot cumulative return
- Show useful summary statistics
- Show a yearly performance table including:
  - annual return
  - Sharpe ratio
  - max drawdown
  - average turnover

## General Requirements

- Write efficient code
- Avoid look-ahead bias
- Keep preprocessing training-only where needed
- Keep the implementation modular and easy to extend
- Use English for code comments and printed messages
